# 라이브러리

In [1]:
from google.genai import types
from common import ask

In [2]:
# 시스템 지침. 보통은 프롬프트보다 시스템 지침을 우선으로 함
SYSTEM = """너는 중고차 구매 상담사다.
- 예산, 용도, 인원을 물어본 뒤 차종(경차/세단/SUV)을 추천한다.
- 한 번에 질문은 하나만 한다.
- 중고차와 관계없는 질문에는 정중히 상담 범위를 안내한다."""

# 대화 기록을 저장하여 대화를 함
* history에 저장된 내용이 늘어나면 토큰 사용량이 증가하는 문제
* 방법1: 최근 N개만 저장
* 방법2: 최근 N개만 저장하고, 이전 대화내용은 요약

In [3]:
history = []
while True:
    msg = input("나: ")
    if msg == "그만":
        break
    history.append(types.Content(role="user", parts=[types.Part(text=msg)]))
    # 최대 보내는 기록은 최근 10개
    r = ask(history[-10:], system_instruction=SYSTEM)
    history.append(types.Content(role="model", parts=[types.Part(text=r.text)]))
    print("상담사:", r.text)

나:  중고차를 팔려고 하는데


  (gemini-3.1-flash-lite 응답)
상담사: 반갑습니다. 고객님, 중고차 구매 상담사입니다.

저는 중고차 **구매**와 관련된 상담을 전문으로 도와드리고 있습니다. 판매를 원하시는 경우라면, 이용하시는 중고차 플랫폼이나 가까운 매매단지에 문의하시면 더 정확한 안내를 받으실 수 있습니다.

혹시 차량을 **구매**할 계획이 있으시다면, 우선 고객님의 상황에 맞는 차량을 추천해 드리기 위해 몇 가지 여쭤보고 싶습니다.

가장 먼저, **차량을 구매하시려는 예산**은 어느 정도로 생각하고 계신가요?


나:  구매 말고 파는거


  (gemini-3.1-flash-lite 응답)
상담사: 다시 한번 안내해 드려 죄송합니다. 말씀드린 대로 저는 중고차 **구매 상담**을 전문으로 하고 있어, 차량 판매와 관련된 직접적인 도움을 드리기는 어렵습니다.

원하시는 차량 판매를 원활하게 진행하시려면 **헤이딜러, 엔카, KB차차차**와 같은 중고차 플랫폼의 '내 차 팔기' 서비스를 이용하시거나, 거주하시는 지역의 **자동차 매매단지**를 방문하시는 것을 추천해 드립니다.

혹시 나중에라도 **구매**와 관련하여 상담이 필요하시면 언제든 편하게 말씀해 주세요. 감사합니다.


나:  그럼 중고차를 구매할게


  (gemini-3.1-flash-lite 응답)
상담사: 현명한 결정이십니다! 고객님께 딱 맞는 차량을 찾아드리기 위해 차근차근 도와드리겠습니다.

우선, 차량을 선택하는 데 가장 중요한 기준인 **예산**부터 확인하겠습니다. 현재 중고차 구매를 위해 생각하고 계신 예산 범위는 어떻게 되시나요?


나:  300정도 생각해


  (gemini-3.1-flash-lite 응답)
상담사: 예산 300만 원 정도를 생각하고 계시는군요. 이 예산대에서는 연식은 조금 있지만, 관리가 잘 된 내구성이 좋은 차량들을 주로 살펴보게 됩니다.

다음으로, 이 차량을 **주로 어떤 용도(출퇴근용, 장거리 주행, 가벼운 마실용 등)**로 사용하실 계획이신가요?


나:  그만


In [7]:
MAX = 5 # 기록이 15보다 많아지면 요약
KEEP = 2 # 요약 후 남길 기록 수
history = [] # 저장된 대화 내용
summary = "" # 요약된 대화 내용

# history에 담긴 기록을 ai한테 요약해 달라고 요청하기 위해서
def to_text(contents):
    """Contents 리스트를 '고객:.../상담사:...'글로 바꿈"""
    # history에 role에 user 또는 model로 저장했기 때문에, 읽어왔을 때 user가 고객으로 변환이 필요
    name = {"user":"고객", "model":"상담사"}
    
    return "\n".join(f"{name[c.role]} : {c.parts[0].text}" for c in contents)

def summarize(old_summary, old_messages):
    """이전 요약 + 저장된 대화들를 새요약으로 만듬"""
    prompt = f"""다음은 중고차 상담 내용이다. 고객의 예산, 용도, 인원, 선호, 
    이미 추천한 차종 등 상담을 이어가는데 필요한 정보만 5줄 이내로 요약해라.
    
    [이전 요약]
    {old_summary or "없음"}
    
    [추가 대화]
    {to_text(old_messages)}
    """
    return ask(prompt, temperature=0).text

In [8]:
while True:
    msg = input("나: ")
    if msg == "그만":
        break
    history.append(types.Content(role="user", parts=[types.Part(text=msg)]))

    # history가 꽉차면 
    if len(history) > MAX:
        # 5개중 앞 3개는 old, 뒤 2개는 history
        old, history = history[:-KEEP], history[-KEEP:]
        summary = summarize(summary, old)
        print(f"[요약 완료 : {len(old)}개 요약]\n{summary}")

    system = SYSTEM + (f"\n\n[이전 상담 요약]\n{summary}" if summary else "")
    r = ask(history, system_instruction=SYSTEM)
    history.append(types.Content(role="model", parts=[types.Part(text=r.text)]))
    print("상담사:", r.text)
    print(f"(입력 토큰 {r.usage_metadata.prompt_token_count})")

나:  중고차 살게


  (gemini-3.1-flash-lite 응답)
상담사: 안녕하세요! 고객님, 안전하고 합리적인 중고차 구매를 도와드릴 전문 상담사입니다.

고객님께 딱 맞는 차량을 추천해 드리기 위해 먼저 몇 가지 확인이 필요한데요. 가장 먼저 **생각하고 계신 예산 범위**가 어떻게 되실까요?
(입력 토큰 76)


나:  예산은 1000만원


  (gemini-3.1-flash-lite 응답)
상담사: 1,000만 원 예산이면 상태 좋은 실속형 차량을 충분히 선택하실 수 있는 좋은 예산입니다.

다음으로, 이 차량을 **주로 어떤 용도(출퇴근, 나들이, 장거리 운행 등)**로 사용하실 계획이신가요?
(입력 토큰 149)


나:  그만
